## <img style="float: left; padding-right: 10px; width: 100px" src="https://raw.githubusercontent.com/sbildea/SEED/main/imagini/unitbv_mie.png"> <center>Știința Explorării și Exploatării Datelor - SEED1</center>


##    <center>Curs 3b: Învățare nesupervizată: asociere și recomandări</center>
**Universitatea Transilvania**<br>
**SIIPA, Semestrul I**<br>
**Instructor:** Conf. dr. Teodor &Scedil;tefan B&icirc;ldea<br>
**Contact:** teodor.bildea@unitbv.ro<br>


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sbildea/SEED/blob/main/cursuri/03-invatare-nesupervizata/curs-03b-asociere-recomandari.ipynb)

In [ ]:
# Configurare curs SEED: rulați această celulă prima.
# Colab: descarcă materialele cursului și instalează pachetele lipsă (aprox. un minut).
# Local: găsește singură rădăcina repository-ului, oricare ar fi folderul curent al kernelului.
import os, sys
MODUL = "03-invatare-nesupervizata"
if "google.colab" in sys.modules:
    if not os.path.exists("/content/SEED"):
        !git clone -q --depth 1 https://github.com/sbildea/SEED.git /content/SEED
        !pip install -q -r /content/SEED/requirements.txt
    RADACINA = "/content/SEED"
else:
    RADACINA = os.path.abspath(".")
    while not os.path.isdir(os.path.join(RADACINA, "seed")) and os.path.dirname(RADACINA) != RADACINA:
        RADACINA = os.path.dirname(RADACINA)
    import importlib.util, socket
    lipsa = [p for p in ("pandas", "matplotlib", "seaborn", "sklearn") if importlib.util.find_spec(p) is None]
    if lipsa:
        print("Lipsesc pachetele", lipsa, "- în terminal: pip install -r requirements-local.txt")
    try:
        socket.create_connection(("raw.githubusercontent.com", 443), 2).close()
    except OSError:
        print("Fără internet: imaginile din text nu se văd. O singură dată, în terminal: python tools/imagini_offline.py")
os.chdir(os.path.join(RADACINA, "cursuri", MODUL))
sys.path.insert(0, RADACINA)

## Cum lucrăm azi

La fel ca la `curs-03a`: **lecția**, **tu** și **agentul tău**. Dacă ai o conversație nouă (sau agentul a rămas fără tokeni), lipește primerul din `curs-03a-pca-clusterizare.ipynb` (secțiunea „Cum lucrăm azi”), apoi *Jurnalul de sesiune* de la finalul notebook-ului unde ai rămas, și scrie „continuă de aici”. Pentru scenele de azi mai lipește și aceste reguli:

```text
Reguli suplimentare pentru coșul de cumpărături (le aplici chiar dacă nu le cer eu):
1. Un coș = o factură. Înainte de reguli: scoate facturile anulate (InvoiceNo începe cu „C”), rândurile cu cantitate <= 0 sau fără descriere și codurile care nu sunt produse (poștă, taxe, ajustări).
2. Lucrează pe o singură țară, alta decât Marea Britanie, ca să încapă în memorie.
3. Spune-mi câte coșuri și câte produse rămân după fiecare pas și ce prag de suport ai folosit, exprimat și în număr de coșuri.
4. Nu-mi prezenta o regulă cu lift mare fără să-mi spui suportul ei.
```

## Problema de azi

Managerul unui magazin online: *„Ce produse se cumpără împreună? Vreau să știu ce să punem alături, ce să oferim la pachet și ce să recomandăm.”*

Aceasta este tot **învățare nesupervizată**: nu avem o țintă, căutăm tipare. Datele sunt tabelul `OnlineRetail` din `lab-set4` (541.909 rânduri, 25.900 de facturi). Un *coș* este o factură.

**Planul de azi (≈50 min):** scena 6 suport, încredere, lift · scena 7 reguli pe date reale · scena 8 recomandări (ghidată, de parcurs singur dacă nu ajungem) · ce îi spunem clientului. Definițiile complete sunt în **Referință**.

## Scena 6. Suport, încredere, lift

**Concept.** O **regulă de asociere** $A \to B$ spune: coșurile care conțin $A$ conțin adesea și $B$. Trei măsuri o descriu:

$$\text{suport}(A) = \frac{\text{coșuri care conțin } A}{\text{toate coșurile}} \qquad
\text{încredere}(A \to B) = \frac{\text{suport}(A \cup B)}{\text{suport}(A)}$$

$$\text{lift}(A \to B) = \frac{\text{încredere}(A \to B)}{\text{suport}(B)} = \frac{\text{suport}(A \cup B)}{\text{suport}(A)\,\text{suport}(B)}$$

- **suport:** cât de des apare tiparul (contează pentru că o regulă rară e zgomot);
- **încredere:** dintre coșurile cu $A$, ce fracțiune au și $B$;
- **lift:** de câte ori cei care au $A$ cumpără $B$ **mai des decât clientul oarecare**. Lift $= 1$ înseamnă independență, $> 1$ asociere, $< 1$ produsele se evită.

*Exemplu clasic.* Din 1.000 de coșuri, 150 conțin burger, 100 ketchup și 50 ambele. Suport(ketchup) = 10%; încredere(burger → ketchup) = 50/150 = 33,3%; lift = 33,3% / 10% = **3,33**: cumpărătorii de burger cumpără ketchup de 3,33 ori mai des decât clientul oarecare.

**Prezice.** Avem șase coșuri mici. Care dintre regulile `pâine → unt`, `lapte → pâine`, `bere → chipsuri` are cel mai mare lift? Ar trebui să acționăm pe ea?

**Cere agentului:**

> Am o listă de 6 coșuri (mulțimi de produse). Calculează suportul, încrederea și lift-ul pentru regulile pâine → unt, lapte → pâine și bere → chipsuri, într-un tabel.

In [ ]:
# Cod de referință: șase coșuri și trei reguli, calculate "la mână"
import pandas as pd

cosuri = [{"pâine", "lapte"}, {"pâine", "unt"}, {"pâine", "lapte", "unt"},
          {"lapte", "bere"}, {"pâine", "lapte", "unt"}, {"bere", "chipsuri"}]
n = len(cosuri)

def suport(produse):
    return sum(set(produse) <= c for c in cosuri) / n

def regula(a, b):
    s_ab = suport(set(a) | set(b))
    incredere = s_ab / suport(a)
    return {"regula": f"{', '.join(a)} → {', '.join(b)}", "suport": s_ab,
            "încredere": incredere, "lift": incredere / suport(b), "coșuri": round(s_ab * n)}

pd.DataFrame([regula({"pâine"}, {"unt"}), regula({"lapte"}, {"pâine"}), regula({"bere"}, {"chipsuri"})]).round(3)

**Verifică.** Control: `pâine → unt`: suport 0,5 (3 coșuri), încredere 0,75, lift 1,5; `lapte → pâine`: suport 0,5, încredere 0,75, lift 1,125; `bere → chipsuri`: suport 0,167 (1 coș), încredere 0,5, lift 3,0.

**Capcană.** Cel mai mare lift (3,0) este la o regulă care se sprijină pe **un singur coș**. Lift mare cu suport mic este de obicei zgomot. De aceea căutăm reguli doar între tiparele cu suport suficient, iar un agent care îți arată regulile „sortate după lift” fără suport te poate duce în eroare (regula 4 din regulile suplimentare).

**Pentru client.** „Spunem că două produse merg împreună numai dacă apar împreună în destule coșuri *și* dacă se cumpără mai des împreună decât ar fi întâmplător.”

## Scena 7. Reguli pe date reale

**Concept.** Cu mii de produse, numărul de combinații explodează. **Apriori** merge pe niveluri: păstrează produsele cu suport mare, apoi perechile formate numai din produse frecvente, apoi tripletele, și așa mai departe. Se sprijină pe o proprietate: *dacă un set de produse nu este frecvent, nici un set care îl conține nu este*. Parametrii tăi: **suportul minim** (exprimat și în număr de coșuri) și **încrederea minimă**; apoi sortăm după lift.

**Prezice.** În tabelul `OnlineRetail`, ce trebuie scos înainte să căutăm reguli? Spune câteva categorii de rânduri. Dacă nu scoatem nimic, ce crezi că vor conține regulile?

**Cere agentului:**

> Pe tabelul `OnlineRetail` din `lab-set4`, pentru o singură țară (France), construiește coșurile (o factură = un coș) și găsește regulile de asociere cu `mlxtend` (suport minim 0,05, încredere minimă 0,5), sortate după lift. Mai întâi fă-o doar scoțând cantitățile ≤ 0, apoi curăță datele (anulări, rânduri fără descriere, coduri care nu sunt produse) și compară numărul de reguli.

In [ ]:
# Cod de referință: ce conține tabelul
from seed.date import incarca
d = incarca("lab-set4")[0]

anulate = d.InvoiceNo.astype(str).str.startswith("C")
print(f"{len(d)} rânduri, {d.InvoiceNo.nunique()} facturi")
print(f"rânduri de pe facturi anulate (prefix C): {anulate.sum()} ({d.loc[anulate, 'InvoiceNo'].nunique()} facturi)")
print(f"cantitate <= 0: {(d.Quantity <= 0).sum()} · fără descriere: {d.Description.isna().sum()}")
print("coduri care nu sunt produse:", d.loc[~d.StockCode.astype(str).str.match(r"^\d{5}"), "StockCode"].value_counts().head(5).to_dict())

In [ ]:
# Cod de referință: coșuri și reguli, înainte și după curățare
from mlxtend.frequent_patterns import apriori, association_rules

def cosuri(df):
    # un rând pe factură, o coloană (True/False) pe produs
    return df.groupby(["InvoiceNo", "Description"]).size().unstack(fill_value=0).gt(0)

def reguli(b, suport_min=0.05, incredere_min=0.5):
    frecvente = apriori(b, min_support=suport_min, use_colnames=True)
    r = association_rules(frecvente, metric="lift", min_threshold=1)
    r = r[r.confidence >= incredere_min].sort_values("lift", ascending=False).copy()
    r["regula"] = r.antecedents.map(lambda s: ", ".join(sorted(s))) + " → " + r.consequents.map(lambda s: ", ".join(sorted(s)))
    return frecvente, r

franta = d[d.Country == "France"]

# varianta grăbită: doar cantități pozitive
b_brut = cosuri(franta[franta.Quantity > 0])
_, r_brut = reguli(b_brut)
cu_post = r_brut.regula.str.contains("POSTAGE").sum()
print(f"BRUT:   {b_brut.shape[0]} coșuri × {b_brut.shape[1]} produse -> {len(r_brut)} reguli, din care {cu_post} conțin POSTAGE")

# varianta curată
produse = franta[~franta.InvoiceNo.astype(str).str.startswith("C") & (franta.Quantity > 0)
                 & franta.Description.notna() & franta.StockCode.astype(str).str.match(r"^\d{5}")].copy()
produse["Description"] = produse.Description.str.strip()
b = cosuri(produse)
frecvente, r = reguli(b)
print(f"CURAT:  {b.shape[0]} coșuri × {b.shape[1]} produse -> {len(frecvente)} seturi frecvente, {len(r)} reguli")
print(f"suport minim 0,05 = cel puțin {int(0.05 * b.shape[0]) + 1} coșuri din {b.shape[0]}")
r[["regula", "support", "confidence", "lift"]].head(8).round(2)

**Verifică.** Control (`France`, suport minim 0,05, încredere minimă 0,5):
- audit: 541.909 rânduri, 25.900 de facturi; **9.288 de rânduri** (3.836 facturi) sunt anulări; 10.624 au cantitate ≤ 0; 1.454 fără descriere; codurile fără sens de produs (`POST`, `DOT`, `M`, `D`, `BANK CHARGES`...);
- BRUT: 392 coșuri × 1.564 produse, **203 reguli, din care 152 conțin `POSTAGE`**;
- CURAT: 382 coșuri × 1.560 produse, 113 seturi frecvente, **51 de reguli**; pragul de 0,05 înseamnă cel puțin 20 de coșuri;
- cele mai puternice: `PACK OF 6 SKULL PAPER CUPS → PACK OF 6 SKULL PAPER PLATES` (suport 0,05, încredere 0,80, lift 13,9) și perechea `CHILDRENS CUTLERY SPACEBOY` ↔ `CHILDRENS CUTLERY DOLLY GIRL` (încredere 0,93 / 0,89, lift 12,6).

**Capcană 1.** Fără curățare, aproape **trei sferturi** din reguli (152 din 203) „descoperă” că oamenii cumpără transport. Un agent nu-și dă seama singur că `POSTAGE` nu este un produs.

**Capcană 2.** Regulile cele mai puternice leagă **variante ale aceluiași produs** (farfurii și pahare din aceeași serie, tacâmuri pentru fetiță și pentru băiat, ceasuri de culori diferite). Sunt reale, dar nu sunt o surpriză pentru manager. O acțiune de marketing valoroasă leagă produse din **categorii diferite**. Întreabă-te: *ce aș face concret cu regula asta?*

**Pentru client.** Pentru fiecare dintre primele trei perechi de produse: o propoziție cu cifrele și o acțiune (pachet, raft vecin, recomandare la finalul comenzii).

In [ ]:
# Din reguli în propoziții pentru manager (o singură direcție pentru fiecare pereche de produse)
unice = r.assign(perechea=r.apply(lambda x: frozenset(x.antecedents | x.consequents), axis=1)).drop_duplicates("perechea")
for _, x in unice.head(3).iterrows():
    print(f"• Dintre coșurile cu [{x.regula.split(' → ')[0]}], {x.confidence:.0%} conțin și [{x.regula.split(' → ')[1]}], "
          f"de {x.lift:.1f} ori mai des decât coșul obișnuit (apare în {x.support:.1%} din coșuri).")

## Scena 8. Recomandări (secțiune ghidată, de parcurs singur dacă nu ajungem)

**Concept.** Regulile de mai sus sunt valabile pentru *toți*. Un **sistem de recomandare** propune lucruri *fiecărui utilizator*. Datele: o matrice utilizator × articol, în care știm doar câteva celule (cine ce a văzut sau a apreciat):
- **feedback explicit** (note, stele): precis, dar rar;
- **feedback implicit** (vizionări, achiziții, clicuri): abundent, dar zgomotos.

Două familii: **pe conținut** (recomandă articole asemănătoare cu cele apreciate; merge și pentru articole noi) și **filtrare colaborativă** (se uită la ce au apreciat utilizatorii asemănători; nu are nevoie de descrieri, dar are nevoie de istoric). Cea mai folosită variantă colaborativă este **factorizarea matricei**: fiecare utilizator $u$ și fiecare articol $i$ primește un vector scurt de **factori latenți**, iar scorul este produsul lor scalar:

$$\hat r_{ui} = p_u^{\top} q_i$$

Aici folosim `TruncatedSVD` din scikit-learn pe setul MovieLens (610 utilizatori, aproape 10.000 de filme), cu feedback implicit: considerăm „apreciat” un film cu nota de cel puțin 4. Împărțim interacțiunile în 80% pentru antrenare și 20% pentru testare. Evaluăm cu **precision@k**: dintre primele $k$ filme recomandate (pe care utilizatorul nu le-a văzut la antrenare), câte apar în setul lui de testare.

**Prezice.** Un model cu 100 de factori latenți va fi mai bun decât unul cu 30? Cu ce referință trebuie comparat orice model de recomandare?

**Cere agentului:**

> Pe fișierul `ml-latest-small/ratings.csv`, construiește matricea utilizator × film cu filmele apreciate (nota ≥ 4), împarte interacțiunile 80/20 (`random_state=17`), antrenează `TruncatedSVD` cu 5, 30 și 100 de componente și compară precision@5, @10 și @20 cu recomandarea „cele mai populare filme”.

In [ ]:
# Cod de referință
import numpy as np
from scipy import sparse
from sklearn.model_selection import train_test_split
from sklearn.decomposition import TruncatedSVD

ratinguri = pd.read_csv("ml-latest-small/ratings.csv")
filme = pd.read_csv("ml-latest-small/movies.csv").set_index("movieId").title

poz = ratinguri[ratinguri.rating >= 4][["userId", "movieId"]]
utilizatori, id_film = poz.userId.astype("category"), poz.movieId.astype("category")
poz = poz.assign(u=utilizatori.cat.codes, i=id_film.cat.codes)
n_u, n_i = int(poz.u.max()) + 1, int(poz.i.max()) + 1

antr, test = train_test_split(poz, test_size=0.2, random_state=17)
matrice = lambda t: sparse.csr_matrix((np.ones(len(t)), (t.u, t.i)), shape=(n_u, n_i))
M_antr, M_test = matrice(antr), matrice(test)
print(f"{n_u} utilizatori, {n_i} filme; {M_antr.nnz} interacțiuni de antrenare, {M_test.nnz} de testare; "
      f"matricea e completată în {M_antr.nnz / (n_u * n_i):.2%}")

def precizie_la_k(scoruri, k):
    s = scoruri.copy()
    s[M_antr.nonzero()] = -np.inf                       # nu recomandăm ce a văzut deja
    top = np.argsort(-s, axis=1)[:, :k]
    relevante = M_test.toarray()
    loviri = np.take_along_axis(relevante, top, axis=1).sum(axis=1)
    return (loviri[relevante.sum(axis=1) > 0] / k).mean()

popularitate = np.tile(np.asarray(M_antr.sum(axis=0)), (n_u, 1)).astype(float)
rez = {"popularitate": [precizie_la_k(popularitate, k) for k in (5, 10, 20)]}
for n_comp in (5, 30, 100):
    svd = TruncatedSVD(n_components=n_comp, random_state=17)
    scoruri = svd.fit_transform(M_antr) @ svd.components_
    rez[f"SVD, {n_comp} factori"] = [precizie_la_k(scoruri, k) for k in (5, 10, 20)]
pd.DataFrame(rez, index=["p@5", "p@10", "p@20"]).T.round(3)

**Verifică.** Control: 609 utilizatori, 6.298 filme; 38.864 interacțiuni de antrenare și 9.716 de testare; matricea e completată în ≈ 1,01%. Precision@5 / @10 / @20:

| Model | p@5 | p@10 | p@20 |
|---|---|---|---|
| popularitate | 0,128 | 0,103 | 0,084 |
| SVD, 5 factori | 0,182 | 0,153 | 0,121 |
| SVD, 30 factori | **0,208** | **0,169** | **0,128** |
| SVD, 100 factori | 0,161 | 0,130 | 0,099 |

**Capcană.** (1) Valorile par mici, dar fiecare utilizator are doar câteva filme în testare dintre peste 6.000: singurul număr cu sens este **comparația cu recomandarea banală** („cele mai populare”). (2) Mai mulți factori **nu** înseamnă mai bine: 100 de factori memorează datele de antrenare (supraînvățare, ca în modulul 4) și coboară spre popularitate. (3) Un utilizator sau un film nou, fără istoric, nu are factori (*cold start*); acolo ajută recomandările pe conținut.

**Pentru client.** „Recomandările personalizate nimeresc de ≈ 1,6 ori mai des decât «cele mai populare» (p@5: 0,208 față de 0,128), dar numai cu o complexitate potrivită a modelului; pentru utilizatorii noi trebuie o altă soluție.”

---
# Referință

Definiții și detalii pentru acasă. Nu se parcurg la curs.

## R1. Alte măsuri pentru reguli

- **Conviction** (diferită de încredere): $\text{conviction}(A \to B) = \dfrac{1 - \text{suport}(B)}{1 - \text{încredere}(A \to B)}$; arată cât de des ar fi greșită regula dacă $A$ și $B$ ar fi independente. Valori mari = regulă care greșește rar.
- **Leverage:** $\text{suport}(A \cup B) - \text{suport}(A)\,\text{suport}(B)$; diferența dintre frecvența observată și cea așteptată la independență.
- **Set frecvent maximal / închis:** un set este *maximal* dacă niciun superset al lui nu este frecvent; *închis* dacă niciun superset nu are același suport. Se folosesc ca să rezumăm mulțimea tiparelor.

Exemplu mic, cu $\text{suport minim} = 3$ coșuri, în figura de mai jos:

<img src="https://raw.githubusercontent.com/sbildea/SEED/main/cursuri/03-invatare-nesupervizata/imagini/frq_itemset_example.png" style="width: 300px">

*Fig. Coșuri și seturi frecvente (Credits [geeksforgeeks](https://www.geeksforgeeks.org/apriori-algorithm/)).*

## R2. Algoritmul Apriori

Propus de Agrawal și Srikant (1994). Caută pe niveluri: seturile frecvente de $k$ produse se folosesc pentru candidații de $k+1$ produse. Se sprijină pe **proprietatea antimonotonă a suportului**: orice submulțime a unui set frecvent este frecventă; echivalent, dacă un set nu este frecvent, nici un superset al lui nu este. Pașii: (1) alegem suportul și încrederea minime; (2) găsim toate seturile cu suport peste prag; (3) din ele formăm regulile cu încredere peste prag; (4) ordonăm după lift.

Limite: cu suport mic și multe produse generează foarte mulți candidați și citește repetat datele; alternativa mai rapidă este **FP-Growth** (`mlxtend.frequent_patterns.fpgrowth`). Regulile descoperite sunt **asocieri, nu cauze**; un număr mare de reguli neinteresante este normal, de aceea filtrăm după lift și suport. Se potrivește mai ușor în B2C (coșuri mari, repetate) decât în B2B (puține comenzi mari, cumpărături planificate).

## R3. Sisteme de recomandare

- **Pe conținut:** articolele și utilizatorii au metadate (gen, vârstă); se recomandă articole apropiate (de exemplu cu k vecini) de cele apreciate. Merge și la pornire la rece, dar nu folosește „înțelepciunea mulțimii”.
- **Colaborativă:** folosește doar matricea de interacțiuni (factorizare, vecini); mai precisă, dar are nevoie de istoric.
- **Hibridă:** combină cele două; bibliotecile hibride populare nu sunt toate întreținute (de exemplu LightFM nu se mai instalează pe versiunile noi de Python).
- **Metrici — explicit:** $\text{MAE} = \frac{1}{N}\sum |r_{ui} - \hat r_{ui}|$ și $\text{RMSE} = \sqrt{\frac{1}{N}\sum (r_{ui} - \hat r_{ui})^2}$.
- **Metrici — implicit:** *precision@k* (dintre primele $k$ recomandate, câte sunt relevante), *recall@k* (dintre cele relevante, câte au ajuns în primele $k$), *MAP*.

**Discuție.** Un sistem care recomandă numai ce seamănă cu ce ai văzut ține utilizatorul într-o „bulă”. Cum am măsura diversitatea recomandărilor? Ce date ale clienților nu ar trebui folosite?

<img src="https://raw.githubusercontent.com/sbildea/SEED/main/cursuri/03-invatare-nesupervizata/imagini/filtering_types.webp" style="width: 520px">

*Fig. Tipuri de sisteme de recomandare (Credits [towardsdatascience](https://towardsdatascience.com/recommender-systems-a-complete-guide-to-machine-learning-models-96d3f94ea748)).*

## R4. Resurse

- Cod complet pe date mici: [extra-03-asociere-recomandari](extra-03-asociere-recomandari.ipynb).
- [Documentația mlxtend despre Apriori](https://rasbt.github.io/mlxtend/user_guide/frequent_patterns/apriori/).
- RecSys, conferința de referință a domeniului: <https://recsys.acm.org>.

---
## Puntea curs → seminar

**Concepte cheie** (modulul 3)

1. **Distanța** între observații și de ce **scalarea** contează.
2. **Corelația** arată ce coloane spun același lucru.
3. **PCA:** comprimăm multe coloane în câteva componente, păstrând cât mai multă varianță; nu folosește ținta.
4. **k-means:** grupuri de observații asemănătoare; K se alege cu cotul și silhouette *și* cu utilitatea; un silhouette mic spune că structura e slabă.
5. **Reguli de asociere:** suport (cât de des), încredere (cât de sigur), lift (cât mai des decât întâmplător); lift mare cu suport mic este zgomot.
6. **Curățarea contează:** ce nu este produs (poștă, taxe, anulări) strică regulile.

**Ce ceri asistentului AI** (exemple de cereri)

> Pentru tabelul `general_data` din `lab-set2`, elimină ID-urile, coloana țintă `Attrition` și coloanele constante, standardizează coloanele numerice, aplică PCA și arată cât din varianță explică primele componente. Apoi aplică k-means pentru K de la 2 la 8 (`random_state=42`, `n_init=10`) și afișează scorul silhouette pentru fiecare K.

> Pentru `lab-set4`, pe o singură țară, curăță datele (anulări, cantități ≤ 0, rânduri fără descriere, coduri care nu sunt produse), construiește coșurile pe facturi și găsește reguli de asociere cu `mlxtend`, sortate după lift. Pentru fiecare regulă arată suportul, încrederea și lift-ul și numărul de coșuri.

**Ce verifici înainte să accepți rezultatul**

- [ ] datele au fost scalate înainte de PCA și k-means;
- [ ] ținta, ID-urile și coloanele constante au fost excluse; știi câte rânduri și coloane au rămas;
- [ ] grupurile sunt interpretate prin mediile coloanelor originale, iar agentul a spus cinstit dacă structura este slabă;
- [ ] coșurile nu conțin anulări, poștă sau alte coduri care nu sunt produse;
- [ ] fiecare regulă are suportul ei (în număr de coșuri) alături de lift;
- [ ] poți spune ce *acțiune* ar urma dintr-o regulă.

Continuăm cu [seminar-03-invatare-nesupervizata.ipynb](seminar-03-invatare-nesupervizata.ipynb).

## Jurnal de sesiune

Completează la sfârșit (sau cere agentului: *„închide scena”*) și păstrează-l în copia ta a notebook-ului. Dacă agentul rămâne fără tokeni, lipește primerul din `curs-03a` și acest jurnal unui alt agent și scrie „continuă de aici”.

- Modul / scena la care am ajuns:
- Agent folosit și data:
- Ce am decis și de ce:
- Valori de control verificate (da / nu / care):
- Greșeli găsite la agent:
- Întrebare deschisă:
- Pentru următorul agent (ce a rămas de făcut, într-o propoziție):